# Explore the PubMed COI database

Connects to the AWS RDS PostgreSQL database (same connection the Vite dev plugin uses) and lets you browse the data with pandas.

Credentials are read from the project's `.env` file. Run the cells top to bottom.

## 1. Install dependencies

Run once. Safe to re-run — pip skips anything already present.

In [14]:
import sys
!{sys.executable} -m pip install -q pandas 'psycopg2-binary>=2.9' sqlalchemy

You should consider upgrading via the '/Users/aisvarya/.pyenv/versions/3.10.3/bin/python -m pip install --upgrade pip' command.


## 2. Load credentials from `.env` and connect

SSL is verified against the RDS CA bundle (`global-bundle.pem`), matching the app.

In [13]:
import os
from pathlib import Path
from sqlalchemy import create_engine, text
import pandas as pd

# Minimal .env parser (no python-dotenv dependency)
def load_env(path='.env'):
    env = {}
    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        k, v = line.split('=', 1)
        env[k.strip()] = v.strip().strip('"').strip("'")
    return env

env = load_env()
ca_path = Path(env.get('DB_SSL_CA', 'global-bundle.pem')).resolve()

engine = create_engine(
    'postgresql+psycopg2://',
    connect_args={
        'host': env['DB_HOST'],
        'port': int(env.get('DB_PORT', 5432)),
        'dbname': env['DB_NAME'],
        'user': env['DB_USER'],
        'password': env['DB_PASSWORD'],
        'sslmode': 'verify-full',
        'sslrootcert': str(ca_path),
    },
)

# Sanity check
with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar())

OperationalError: (psycopg2.OperationalError) connection to server at "pubmed-coi-db.cnvthm1pgcw1.us-east-2.rds.amazonaws.com" (3.14.193.229), port 5432 failed: root certificate file "/Users/aisvarya/Documents/Documents - MacBook Pro (67)/tow/theExplainer/energy drinks/pubmed-semantic-interactive copy/entity_dedup_experiments/global-bundle.pem" does not exist
Either provide the file, use the system's trusted roots with sslrootcert=system, or change sslmode to disable server certificate verification.

(Background on this error at: https://sqlalche.me/e/20/e3q8)

## 3. What tables exist?

In [ ]:
tables = pd.read_sql(text("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name
"""), engine)

# Row count per table
counts = []
with engine.connect() as conn:
    for t in tables['table_name']:
        n = conn.execute(text(f'SELECT count(*) FROM "{t}"')).scalar()
        counts.append({'table': t, 'rows': n})
pd.DataFrame(counts)

,table,rows
0,article,16383
1,article_author,89495
2,article_citation,224349
3,article_embedding,16383
4,article_favorability,0
5,article_funding,11725
6,article_keyword,181891
7,author,42628
8,author_affiliation,100713
9,author_coi,54208


## 4. Peek at any table

Change `TABLE` to any name from the list above.

In [ ]:
TABLE = 'article'
pd.read_sql(text(f'SELECT * FROM "{TABLE}" LIMIT 20'), engine)

,article_id,pmid,doi,title,abstract,conclusion,reasoning,journal,pubmed_url,publication_year,publication_month,publication_day
0,4,34467135,10.1371/journal.pone.0204416,Vaping Among Delaware Youth.,,None,None,Delaware journal of public health,https://pubmed.ncbi.nlm.nih.gov/34467135/,2020,Aug,NaN
1,5,34423079,10.18001/trs.6.6.5,"Variable Voltage, Tank-Style ENDS Do Not Alway...",This study's objective was to characterize the...,None,None,Tobacco regulatory science,https://pubmed.ncbi.nlm.nih.gov/34423079/,2020,Nov,NaN
2,6,34394241,10.4314/ahs.v20i4.33,Acute effects of electronic cigarette smoking ...,Electronic cigarette (e-cigarette) use is cons...,None,None,African health sciences,https://pubmed.ncbi.nlm.nih.gov/34394241/,2020,Dec,NaN
3,18,33870313,10.34197/ats-scholar.2020-0022RE,ATS Core Curriculum 2020. Pediatric Pulmonary ...,The American Thoracic Society Core Curriculum ...,None,None,ATS scholar,https://pubmed.ncbi.nlm.nih.gov/33870313/,2020,Dec,30.0
4,7,34345845,10.1016/j.toxlet.2012.01.006,The use of human induced pluripotent stem cell...,devTOX quickPredict (devTOX qP ) is a metabolo...,None,None,Current research in toxicology,https://pubmed.ncbi.nlm.nih.gov/34345845/,2020,Jun,10.0
5,8,34336231,10.1164/rccm.201912-2332WS,Loose ENDs: Electronic Nicotine Delivery Syste...,,None,None,European medical journal. Respiratory,https://pubmed.ncbi.nlm.nih.gov/34336231/,2020,Nov,NaN
6,9,34321706,10.1016/j.jpsychires.2015.02.008,Personality and impulsivity as predictors of t...,The tobacco industry markets their products to...,None,None,Personality and individual differences,https://pubmed.ncbi.nlm.nih.gov/34321706/,2020,Sep,1.0
7,10,34321700,10.1111/jora.12454,"Sexual Orientation-Based Alcohol, Tobacco, and...",This study investigated whether the presence o...,None,None,Youth & society,https://pubmed.ncbi.nlm.nih.gov/34321700/,2020,Oct,NaN
8,11,34007628,10.1016/j.ypmed.2018.07.002,Attitudes and Perceptions of Tobacco-Related P...,Despite the highly publicized health consequen...,None,None,Innovations in pharmacy,https://pubmed.ncbi.nlm.nih.gov/34007628/,2020,None,NaN
9,15,33994818,10.1177/1179173X20953402,Perceptions and Use of Electronic Nicotine Del...,Rapid increase in youth use of Electronic Nico...,None,None,Tobacco use insights,https://pubmed.ncbi.nlm.nih.gov/33994818/,2020,None,NaN


## 5. The denormalized article view

This is the exact flattened shape the app consumes (see `db-export.mjs`): one row per article with authors, keywords, COIs, funding, affiliations and 2-D embedding coords joined in.

In [ ]:
# pull all the journal titles from the article table
journals = pd.read_sql(text(f'SELECT DISTINCT journal FROM "{TABLE}"'), engine)
journals

,journal
0,Journal of biomedical optics
1,Archivos de la Sociedad Espanola de Oftalmologia
2,Journal of visualized experiments : JoVE
3,Analytical and bioanalytical chemistry
4,"Urologiia (Moscow, Russia : 1999)"
...,...
2124,Frontiers in toxicology
2125,American journal of respiratory and critical c...
2126,International medical case reports journal
2127,Journal of addictions nursing


In [ ]:
# remove duplicates and sort the journal titles
journals = journals.drop_duplicates().sort_values(by='journal').reset_index(drop=True)
journals

,journal
0,A&A practice
1,AACE endocrinology and diabetes
2,AANA journal
3,ACS applied bio materials
4,ACS central science
...,...
2124,iScience
2125,jLPHA : the official journal of the Louisiana ...
2126,mBio
2127,medRxiv : the preprint server for health sciences


In [ ]:
SQL = """
  SELECT
    a.pmid,
    a.title,
    a.journal,
    a.publication_year,
    e.x, e.y,
    (SELECT string_agg(au.full_name, '; ' ORDER BY aa.author_order)
       FROM article_author aa JOIN author au ON au.author_id = aa.author_id
       WHERE aa.article_id = a.article_id) AS authors,
    (SELECT string_agg(k.keyword_text, '; ')
       FROM article_keyword ak JOIN keyword k ON k.keyword_id = ak.keyword_id
       WHERE ak.article_id = a.article_id) AS keywords,
    (SELECT string_agg(DISTINCT NULLIF(btrim(co.coi_institution), ''), '; ')
       FROM author_coi co WHERE co.article_id = a.article_id) AS coi_org,
    (SELECT string_agg(DISTINCT i.name, '; ')
       FROM article_funding fu JOIN institution i ON i.institution_id = fu.institution_id
       WHERE fu.article_id = a.article_id) AS funding
  FROM article a
  JOIN article_embedding e ON e.article_id = a.article_id
"""
df = pd.read_sql(text(SQL), engine)
print(f'{len(df)} articles')
df.head(20)

16383 articles


,pmid,title,journal,publication_year,x,y,authors,keywords,coi_org,funding
0,34467135,Vaping Among Delaware Youth.,Delaware journal of public health,2020.0,16.402185,56.844009,Rachel Ryding; David Borton; Meisje Scales; CP...,None,None,None
1,34423079,"Variable Voltage, Tank-Style ENDS Do Not Alway...",Tobacco regulatory science,2020.0,-29.824451,-6.744315,Alisha Eversole; Sarah Maloney; Soha Talih; Ro...,ENDS; nicotine delivery,Tobacco Industry; Electronic Cigarette Industry,NIDA NIH HHS
2,34394241,Acute effects of electronic cigarette smoking ...,African health sciences,2020.0,-43.335728,-19.039913,Vahit Demir; Siho Hidayet; Yaşar Turan; Hüseyi...,Electronic Nicotine Delivery Systems; Humans; ...,None,None
3,33870313,ATS Core Curriculum 2020. Pediatric Pulmonary ...,ATS scholar,2020.0,-19.036497,8.196666,Jane E Gross; Michael Y McCown; Caroline Okori...,e-cigarettes; pediatric; review; sarcoidosis; ...,None,None
4,34345845,The use of human induced pluripotent stem cell...,Current research in toxicology,2020.0,-57.798988,-0.141076,Liam Simms; Kathryn Rudd; Jessica Palmer; Luka...,"ATRA, All-trans-retinoic acid; CDC, Centers fo...",Imperial Brands PLC,None
5,34336231,Loose ENDs: Electronic Nicotine Delivery Syste...,European medical journal. Respiratory,2020.0,-3.300706,23.497181,Saira Ahmad; M Flori Sassano; Robert Tarran,Airway; electronic cigarette (e-cigarette); lu...,None,NHLBI NIH HHS
6,34321706,Personality and impulsivity as predictors of t...,Personality and individual differences,2020.0,39.208847,-55.322155,Jenny E Ozga-Hess; Katelyn F Romm; Nicholas J ...,Alternative tobacco product; College student; ...,None,ACL HHS; NCCDPHP CDC HHS; NIGMS NIH HHS
7,34321700,"Sexual Orientation-Based Alcohol, Tobacco, and...",Youth & society,2020.0,30.271051,-76.843178,Lei Zhang; Laura J Finan; Melina Bersamin; Deb...,adolescent substance use; health disparities; ...,None,NIAAA NIH HHS; NICHD NIH HHS
8,34007628,Attitudes and Perceptions of Tobacco-Related P...,Innovations in pharmacy,2020.0,76.379715,-23.135063,Yen H Dang,electronic cigarettes; cigarettes; cigars; col...,None,None
9,33994818,Perceptions and Use of Electronic Nicotine Del...,Tobacco use insights,2020.0,1.422284,-20.217882,Anastasiya Ferrell; Linda Hadddad; Jennifer Ha...,electronic cigarettes; Electronic nicotine del...,None,None


## 6. Ad-hoc queries

`df` from the cell above is a normal DataFrame — slice it, or run fresh SQL below. Examples:

In [ ]:
# Most common COI organizations
pd.read_sql(text("""
    SELECT btrim(coi_institution) AS coi_org, count(DISTINCT article_id) AS articles
    FROM author_coi
    WHERE NULLIF(btrim(coi_institution), '') IS NOT NULL
    GROUP BY 1
    ORDER BY articles DESC
    LIMIT 25
"""), engine)

,coi_org,articles
0,Pfizer,69
1,Pfizer; Johnson & Johnson,52
2,"Pfizer, Inc.",35
3,Pfizer; J&J,33
4,Tobacco Industry; Electronic Cigarette Industry,33
5,National Institutes of Health,33
6,Governments,24
7,Tobacco Industry,21
8,Vaping Industry,20
9,GSK; Pfizer; Novartis; J&J; Cypress Bioscience,18


In [ ]:
# --- 1. Author affiliations (author -> institution, per article) ---
affiliations = pd.read_sql(text("""
    SELECT au.full_name AS author, i.name AS institution, a.pmid, a.title
    FROM author_affiliation af
    JOIN author au      ON au.author_id = af.author_id
    JOIN institution i  ON i.institution_id = af.institution_id
    JOIN article a      ON a.article_id = af.article_id
    ORDER BY af.article_id, au.full_name
    LIMIT 25
"""), engine)
affiliations['institution'][0:10].to_list()

['Núcleo de Avaliação de Tecnologias em Saúde, Divisão de Pesquisa Populacional, Instituto Nacional de Câncer José Alencar Gomes da Silva (INCA). Praça Cruz Vermelha 23, Centro. 20230-130 Rio de Janeiro RJ Brasil. lauraabarufaldi@gmail.com.',
 'Núcleo de Avaliação de Tecnologias em Saúde, Divisão de Pesquisa Populacional, Instituto Nacional de Câncer José Alencar Gomes da Silva (INCA). Praça Cruz Vermelha 23, Centro. 20230-130 Rio de Janeiro RJ Brasil. lauraabarufaldi@gmail.com.',
 'Coordenação de Prevenção e Vigilância, INCA. Rio de Janeiro RJ Brasil.',
 'Divisão de Pesquisa Populacional, INCA. Rio de Janeiro RJ Brasil.',
 'Núcleo de Avaliação de Tecnologias em Saúde, Divisão de Pesquisa Populacional, Instituto Nacional de Câncer José Alencar Gomes da Silva (INCA). Praça Cruz Vermelha 23, Centro. 20230-130 Rio de Janeiro RJ Brasil. lauraabarufaldi@gmail.com.',
 'Núcleo de Avaliação de Tecnologias em Saúde, Divisão de Pesquisa Populacional, Instituto Nacional de Câncer José Alencar Gom

In [ ]:
affiliations = pd.read_sql(text("""
    SELECT au.full_name AS author, i.name AS institution, a.pmid, a.title
    FROM author_affiliation af
    JOIN author au      ON au.author_id = af.author_id
    JOIN institution i  ON i.institution_id = af.institution_id
    JOIN article a      ON a.article_id = af.article_id
    WHERE i.name ILIKE '%Enthalpy Analytical%'
    ORDER BY af.article_id, au.full_name
"""), engine)   

affiliations['institution'].to_list()

['Enthalpy Analytical, Durham, NC, USA.',
 'Enthalpy Analytical, Durham, NC, USA. Electronic address: gene.gillman@enthalpy.com.',
 'Enthalpy Analytical, Durham, NC, USA.',
 'Enthalpy Analytical, Inc., Durham, NC, United States.',
 'Enthalpy Analytical, Incorporated, Durham, North Carolina.',
 'Enthalpy Analytical, Inc., 800 Capitola Drive, Suite 1, Durham, NC 27713, USA.',
 'Enthalpy Analytical, Inc., 800 Capitola Drive, Suite 1, Durham, NC 27713, USA.',
 'Enthalpy Analytical Inc., 800 Capitola Drive, Durham, NC 27713, USA. Gene.Gillman@enthalpy.com.',
 'Enthalpy Analytical, Inc., 800 Capitola Drive, Suite 1, Durham, North Carolina 27713, United States.',
 'Enthalpy Analytical Inc., Durham, NC 27713, USA.',
 'Enthalpy Analytical Inc., Durham, NC 27713, USA.',
 'Enthalpy Analytical Inc., Durham, NC 27713, USA. Electronic address: gene.gillman@enthalpy.com.',
 'Enthalpy Analytical, Inc., 800 Capitola Drive, Suite 1, NC 27713, USA. gene.gillman@enthalpy.com.',
 'Enthalpy Analytical Inc.,

In [ ]:
affiliations['institution'].unique()

array(['Juul Labs, Inc., United States.',
       'Juul Labs, Inc., United States. Electronic address: nicholas.goldenson@juul.com.',
       'Juul Labs, Inc.', 'Juul Labs, Inc., San Francisco, USA.',
       'JUUL Labs, Inc, Washington, DC, USA.',
       'Juul Labs, Inc, Washington, DC, United States.',
       'Regulatory Sciences, Juul Labs, Inc., Washington, DC 20004, USA.',
       'Department of Regulatory Sciences, Juul Labs, Inc., Washington, DC 20004, USA.',
       'Regulatory Sciences, Juul Labs, Inc., Washington DC 20004, USA.',
       'Juul Labs, Inc, Washington, DC, USA.',
       'Juul Labs Inc., San Francisco, CA, USA.',
       'Juul Labs, Inc., Washington, DC, USA.',
       'Juul Labs, Inc. Washington, DC, USA.',
       'JUUL Labs, Inc, Washington, District of Columbia, USA.',
       'Juul Labs, Inc, United States. Electronic address: Nicholas.Goldenson@Juul.com.',
       'Juul Labs, Inc, United States.',
       'Behavioral and Clinical Sciences, Juul Labs, Inc, Washington, D

In [2]:
import re, collections
import pandas as pd
import dedupe

ImportError: Numba needs NumPy 2.0 or less. Got NumPy 2.2.

In [ ]:
top_funders = pd.read_sql(text("""
    SELECT i.name AS funder, count(DISTINCT fu.article_id) AS articles
    FROM article_funding fu
    JOIN institution i ON i.institution_id = fu.institution_id
    GROUP BY 1 ORDER BY articles DESC LIMIT 20
"""), engine)
print(top_funders.to_string(index=False))

                  funder  articles
             NCI NIH HHS      2431
            NIDA NIH HHS      2257
           NHLBI NIH HHS       890
           NIEHS NIH HHS       540
           NCATS NIH HHS       443
           NIGMS NIH HHS       342
      Cancer Research UK       310
Medical Research Council       280
                 NIH HHS       264
           NIAAA NIH HHS       232
           NIMHD NIH HHS       185
           NICHD NIH HHS       167
                    CIHR       148
             FIC NIH HHS       100
    Department of Health        97
           NIDDK NIH HHS        90
      Intramural CDC HHS        82
            NIMH NIH HHS        81
                 FDA HHS        70
             NIA NIH HHS        65
